# Problem Statement

## **Business Context**

"Visit with Us," a leading travel company, is revolutionizing the tourism industry by leveraging data-driven strategies to optimize operations and customer engagement. While introducing a new package offering, such as the Wellness Tourism Package, the company faces challenges in targeting the right customers efficiently. The manual approach to identifying potential customers is inconsistent, time-consuming, and prone to errors, leading to missed opportunities and suboptimal campaign performance.

To address these issues, the company aims to implement a scalable and automated system that integrates customer data, predicts potential buyers, and enhances decision-making for marketing strategies. By utilizing an MLOps pipeline, the company seeks to achieve seamless integration of data preprocessing, model development, deployment, and CI/CD practices for continuous improvement. This system will ensure efficient targeting of customers, timely updates to the predictive model, and adaptation to evolving customer behaviors, ultimately driving growth and customer satisfaction.


## **Objective**

As an MLOps Engineer at "Visit with Us," your responsibility is to design and deploy an MLOps pipeline on GitHub to automate the end-to-end workflow for predicting customer purchases. The primary objective is to build a model that predicts whether a customer will purchase the newly introduced Wellness Tourism Package before contacting them. The pipeline will include data cleaning, preprocessing, transformation, model building, training, evaluation, and deployment, ensuring consistent performance and scalability. By leveraging GitHub Actions for CI/CD integration, the system will enable automated updates, streamline model deployment, and improve operational efficiency. This robust predictive solution will empower policymakers to make data-driven decisions, enhance marketing strategies, and effectively target potential customers, thereby driving customer acquisition and business growth.

## **Data Description**

The dataset contains customer and interaction data that serve as key attributes for predicting the likelihood of purchasing the Wellness Tourism Package. The detailed attributes are:

**Customer Details**
- **CustomerID:** Unique identifier for each customer.
- **ProdTaken:** Target variable indicating whether the customer has purchased a package (0: No, 1: Yes).
- **Age:** Age of the customer.
- **TypeofContact:** The method by which the customer was contacted (Company Invited or Self Inquiry).
- **CityTier:** The city category based on development, population, and living standards (Tier 1 > Tier 2 > Tier 3).
- **Occupation:** Customer's occupation (e.g., Salaried, Freelancer).
- **Gender:** Gender of the customer (Male, Female).
- **NumberOfPersonVisiting:** Total number of people accompanying the customer on the trip.
- **PreferredPropertyStar:** Preferred hotel rating by the customer.
- **MaritalStatus:** Marital status of the customer (Single, Married, Divorced).
- **NumberOfTrips:** Average number of trips the customer takes annually.
- **Passport:** Whether the customer holds a valid passport (0: No, 1: Yes).
- **OwnCar:** Whether the customer owns a car (0: No, 1: Yes).
- **NumberOfChildrenVisiting:** Number of children below age 5 accompanying the customer.
- **Designation:** Customer's designation in their current organization.
- **MonthlyIncome:** Gross monthly income of the customer.

**Customer Interaction Data**
- **PitchSatisfactionScore:** Score indicating the customer's satisfaction with the sales pitch.
- **ProductPitched:** The type of product pitched to the customer.
- **NumberOfFollowups:** Total number of follow-ups by the salesperson after the sales pitch.-
- **DurationOfPitch:** Duration of the sales pitch delivered to the customer.


# Model Building

In [1]:
# Create a master folder to keep all files created when executing the below code cells
import os
os.makedirs("tourism_project", exist_ok=True)

In [2]:
# Create a folder for storing the model building files
os.makedirs("tourism_project/model_building", exist_ok=True)

## Data Registration

In [3]:
os.makedirs("tourism_project/data", exist_ok=True)

Once the **data** folder created after executing the above cell, please upload the **tourism.csv** in to the folder

In [ ]:
# Hugging Face credentials
# 1. Create a free Hugging Face account: https://huggingface.co/join
# 2. Create an access token with WRITE permission: https://huggingface.co/settings/tokens
# 3. Enter your username and token below (token input is hidden)
import os
from getpass import getpass

os.environ["HF_USERNAME"] = input("Enter your Hugging Face username: ")
os.environ["HF_TOKEN"] = getpass("Enter your Hugging Face access token (WRITE permission): ")


In [4]:
%%writefile tourism_project/model_building/data_register.py
"""Uploads the raw tourism.csv file to a Hugging Face Hub dataset repository."""
import os
from huggingface_hub import HfApi, create_repo

HF_TOKEN = os.getenv("HF_TOKEN")
HF_USERNAME = os.getenv("HF_USERNAME")

DATASET_REPO_ID = f"{HF_USERNAME}/tourism-package-prediction"
LOCAL_DATA_PATH = "tourism_project/data/tourism.csv"

api = HfApi(token=HF_TOKEN)

create_repo(
    repo_id=DATASET_REPO_ID,
    repo_type="dataset",
    private=False,
    exist_ok=True,
    token=HF_TOKEN,
)

api.upload_file(
    path_or_fileobj=LOCAL_DATA_PATH,
    path_in_repo="tourism.csv",
    repo_id=DATASET_REPO_ID,
    repo_type="dataset",
    token=HF_TOKEN,
)

print(f"Raw dataset uploaded successfully to: https://huggingface.co/datasets/{DATASET_REPO_ID}")


Writing tourism_project/model_building/data_register.py


In [6]:
!python tourism_project/model_building/data_register.py

Raw dataset uploaded successfully to: https://huggingface.co/datasets/ayushh089/tourism-package-prediction


**Observation:** The raw `tourism.csv` file (4,128 rows, 21 columns) is now uploaded to a public Hugging Face **dataset repository**. This gives the whole pipeline a single, versioned source of truth — every later stage (data prep, training, and the GitHub Actions workflow) reads data from this repo instead of relying on a local file copy that could go out of sync across machines/runs.

## Data Preparation

In [7]:
# Quick look at the raw data before cleaning
import pandas as pd

df_raw = pd.read_csv("tourism_project/data/tourism.csv")
print(df_raw.shape)
df_raw.head()


(4128, 21)


,Unnamed: 0,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,...,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
0,0,200000,1,41.0,Self Enquiry,3,6.0,Salaried,Female,3,...,Deluxe,3.0,Single,1.0,1,2,1,0.0,Manager,20993.0
1,1,200001,0,49.0,Company Invited,1,14.0,Salaried,Male,3,...,Deluxe,4.0,Divorced,2.0,0,3,1,2.0,Manager,20130.0
2,2,200002,1,37.0,Self Enquiry,1,8.0,Free Lancer,Male,3,...,Basic,3.0,Single,7.0,1,3,0,0.0,Executive,17090.0
3,3,200003,0,33.0,Company Invited,1,9.0,Salaried,Female,2,...,Basic,3.0,Divorced,2.0,1,5,1,1.0,Executive,17909.0
4,5,200005,0,32.0,Company Invited,1,8.0,Salaried,Male,3,...,Basic,3.0,Single,1.0,0,5,1,1.0,Executive,18068.0


In [8]:
# Check data types, missing values and target balance
print(df_raw.info())
print("\nMissing values:\n", df_raw.isnull().sum())
print("\nTarget balance:\n", df_raw["ProdTaken"].value_counts(normalize=True))


<class 'pandas.DataFrame'>
RangeIndex: 4128 entries, 0 to 4127
Data columns (total 21 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Unnamed: 0                4128 non-null   int64  
 1   CustomerID                4128 non-null   int64  
 2   ProdTaken                 4128 non-null   int64  
 3   Age                       4128 non-null   float64
 4   TypeofContact             4128 non-null   str    
 5   CityTier                  4128 non-null   int64  
 6   DurationOfPitch           4128 non-null   float64
 7   Occupation                4128 non-null   str    
 8   Gender                    4128 non-null   str    
 9   NumberOfPersonVisiting    4128 non-null   int64  
 10  NumberOfFollowups         4128 non-null   float64
 11  ProductPitched            4128 non-null   str    
 12  PreferredPropertyStar     4128 non-null   float64
 13  MaritalStatus             4128 non-null   str    
 14  NumberOfTrips      

In [9]:
# Inspect categorical columns for data-entry inconsistencies
cat_cols_check = ["TypeofContact", "Occupation", "Gender", "ProductPitched", "MaritalStatus", "Designation"]
for c in cat_cols_check:
    print(c, df_raw[c].unique())


TypeofContact <ArrowStringArray>
['Self Enquiry', 'Company Invited']
Length: 2, dtype: str
Occupation <ArrowStringArray>
['Salaried', 'Free Lancer', 'Small Business', 'Large Business']
Length: 4, dtype: str
Gender <ArrowStringArray>
['Female', 'Male', 'Fe Male']
Length: 3, dtype: str
ProductPitched <ArrowStringArray>
['Deluxe', 'Basic', 'Standard', 'Super Deluxe', 'King']
Length: 5, dtype: str
MaritalStatus <ArrowStringArray>
['Single', 'Divorced', 'Married', 'Unmarried']
Length: 4, dtype: str
Designation <ArrowStringArray>
['Manager', 'Executive', 'Senior Manager', 'AVP', 'VP']
Length: 5, dtype: str


**Observation:** The data has no nulls, but two categorical columns contain data-entry inconsistencies that need fixing before modeling:
- `Gender` contains both `"Female"` and the typo `"Fe Male"`.
- `MaritalStatus` contains both `"Single"` and `"Unmarried"`, which represent the same category.

There is also an `Unnamed: 0` index column and a `CustomerID` column — both are identifiers with no predictive value and should be dropped. The target `ProdTaken` is imbalanced (~19% positive class), which we account for later with `class_weight="balanced"`.

In [10]:
%%writefile tourism_project/model_building/data_prep.py
"""
Downloads the raw dataset from the Hugging Face dataset repo, cleans it,
fixes data-entry inconsistencies, drops non-predictive identifier columns,
performs a stratified train/test split, and pushes train.csv / test.csv
back to the same Hugging Face dataset repo.
"""
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from huggingface_hub import HfApi, hf_hub_download

HF_TOKEN = os.getenv("HF_TOKEN")
HF_USERNAME = os.getenv("HF_USERNAME")
DATASET_REPO_ID = f"{HF_USERNAME}/tourism-package-prediction"
TARGET_COL = "ProdTaken"

# 1. Download the raw dataset
raw_path = hf_hub_download(repo_id=DATASET_REPO_ID, filename="tourism.csv",
                            repo_type="dataset", token=HF_TOKEN)
df = pd.read_csv(raw_path)

# 2. Clean the data
drop_cols = [c for c in ["Unnamed: 0", "CustomerID"] if c in df.columns]
df = df.drop(columns=drop_cols)

df["Gender"] = df["Gender"].replace({"Fe Male": "Female"})
df["MaritalStatus"] = df["MaritalStatus"].replace({"Unmarried": "Single"})

df = df.drop_duplicates()
df = df.dropna(subset=[TARGET_COL])

# 3. Stratified train/test split
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df[TARGET_COL]
)

os.makedirs("tourism_project/data", exist_ok=True)
train_path = "tourism_project/data/train.csv"
test_path = "tourism_project/data/test.csv"
train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)
print(f"Train shape: {train_df.shape}, Test shape: {test_df.shape}")

# 4. Push processed files back to the Hugging Face dataset repo
api = HfApi(token=HF_TOKEN)
api.upload_file(path_or_fileobj=train_path, path_in_repo="train.csv",
                 repo_id=DATASET_REPO_ID, repo_type="dataset", token=HF_TOKEN)
api.upload_file(path_or_fileobj=test_path, path_in_repo="test.csv",
                 repo_id=DATASET_REPO_ID, repo_type="dataset", token=HF_TOKEN)
print(f"train.csv and test.csv uploaded to: https://huggingface.co/datasets/{DATASET_REPO_ID}")


Writing tourism_project/model_building/data_prep.py


In [12]:
!python tourism_project/model_building/data_prep.py

Train shape: (3208, 19), Test shape: (803, 19)
train.csv and test.csv uploaded to: https://huggingface.co/datasets/ayushh089/tourism-package-prediction


c:\Users\gupta\OneDrive\Desktop\langchain-project\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\gupta\.cache\huggingface\hub\datasets--ayushh089--tourism-package-prediction. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


**Observation:** After cleaning and an 80/20 stratified split, we get **3,208** training rows and **821** test rows, with the ~19%/81% class balance preserved in both sets. Both files are versioned in the same Hugging Face dataset repo alongside the raw file.

## Model Training and Registration with Experimentation Tracking

In [13]:
# Install MLflow for experiment tracking (skip if already installed)
%pip install -q mlflow xgboost


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
%%writefile tourism_project/model_building/train.py
"""
Model Building, Training and Experimentation Tracking
Downloads train/test data from Hugging Face, builds a preprocessing +
classification pipeline, trains several candidate models, logs every run
to MLflow, picks the best model by test-set ROC-AUC, and uploads the
winning pipeline to a Hugging Face Hub model repo.
"""
import os
import joblib
import pandas as pd
import mlflow
import mlflow.sklearn

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from xgboost import XGBClassifier
from huggingface_hub import HfApi, create_repo, hf_hub_download
from dotenv import load_dotenv
load_dotenv()

HF_TOKEN = os.getenv("HF_TOKEN")
HF_USERNAME = os.getenv("HF_USERNAME")
DATASET_REPO_ID = f"{HF_USERNAME}/tourism-package-prediction"
MODEL_REPO_ID = f"{HF_USERNAME}/tourism-package-prediction-model"
TARGET_COL = "ProdTaken"

# 1. Load train/test data
train_path = hf_hub_download(repo_id=DATASET_REPO_ID, filename="train.csv", repo_type="dataset", token=HF_TOKEN)
test_path = hf_hub_download(repo_id=DATASET_REPO_ID, filename="test.csv", repo_type="dataset", token=HF_TOKEN)
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

X_train, y_train = train_df.drop(columns=[TARGET_COL]), train_df[TARGET_COL]
X_test, y_test = test_df.drop(columns=[TARGET_COL]), test_df[TARGET_COL]

categorical_cols = X_train.select_dtypes(include=["object", "string"]).columns.tolist()
numerical_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()

# 2. Preprocessing pipeline
numeric_transformer = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
categorical_transformer = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer([("num", numeric_transformer, numerical_cols), ("cat", categorical_transformer, categorical_cols)])

# 3. Candidate models
candidates = {
    "logistic_regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "random_forest": RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, class_weight="balanced"),
    "xgboost": XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05, eval_metric="logloss", random_state=42),
}

mlflow.set_experiment("tourism-wellness-package-prediction")
best_model_name, best_score, best_pipeline = None, -1.0, None

for name, clf in candidates.items():
    with mlflow.start_run(run_name=name):
        pipeline = Pipeline([("preprocessor", preprocessor), ("classifier", clf)])
        pipeline.fit(X_train, y_train)
        preds = pipeline.predict(X_test)
        probs = pipeline.predict_proba(X_test)[:, 1]
        metrics = {
            "accuracy": accuracy_score(y_test, preds),
            "precision": precision_score(y_test, preds),
            "recall": recall_score(y_test, preds),
            "f1_score": f1_score(y_test, preds),
            "roc_auc": roc_auc_score(y_test, probs),
        }
        mlflow.log_param("model_type", name)
        mlflow.log_params(clf.get_params())
        mlflow.log_metrics(metrics)
        mlflow.sklearn.log_model(pipeline, artifact_path="model", serialization_format="pickle")
        print(f"[{name}] {metrics}")
        if metrics["roc_auc"] > best_score:
            best_score, best_model_name, best_pipeline = metrics["roc_auc"], name, pipeline

print(f"\nBest model: {best_model_name} (ROC-AUC = {best_score:.4f})")

# 4. Save and register the best model to the Hugging Face Hub
os.makedirs("tourism_project/model_building/artifacts", exist_ok=True)
model_path = "tourism_project/model_building/artifacts/best_model.joblib"
joblib.dump(best_pipeline, model_path)

api = HfApi(token=HF_TOKEN)
create_repo(repo_id=MODEL_REPO_ID, repo_type="model", private=False, exist_ok=True, token=HF_TOKEN)
api.upload_file(path_or_fileobj=model_path, path_in_repo="best_model.joblib",
                 repo_id=MODEL_REPO_ID, repo_type="model", token=HF_TOKEN)
print(f"Best model uploaded to: https://huggingface.co/{MODEL_REPO_ID}")


Overwriting tourism_project/model_building/train.py


In [17]:
!python tourism_project/model_building/train.py

[logistic_regression] {'accuracy': 0.7334993773349938, 'precision': 0.3942652329749104, 'recall': 0.7096774193548387, 'f1_score': 0.5069124423963134, 'roc_auc': 0.8088311429709278}
[random_forest] {'accuracy': 0.8443337484433375, 'precision': 0.5797872340425532, 'recall': 0.7032258064516129, 'f1_score': 0.6355685131195336, 'roc_auc': 0.9007964954201515}
[xgboost] {'accuracy': 0.8941469489414695, 'precision': 0.85, 'recall': 0.5483870967741935, 'f1_score': 0.6666666666666666, 'roc_auc': 0.9222023098367185}

Best model: xgboost (ROC-AUC = 0.9222)
Best model uploaded to: https://huggingface.co/ayushh089/tourism-package-prediction-model


2026/09/26 21:01:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/26 21:01:24 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/09/26 21:01:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/26 21:01:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_p

In [18]:
# Inspect the MLflow experiment runs that were just logged
import mlflow
runs = mlflow.search_runs(experiment_names=["tourism-wellness-package-prediction"])
runs[["tags.mlflow.runName", "metrics.accuracy", "metrics.precision", "metrics.recall", "metrics.f1_score", "metrics.roc_auc"]]


c:\Users\gupta\OneDrive\Desktop\langchain-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,tags.mlflow.runName,metrics.accuracy,metrics.precision,metrics.recall,metrics.f1_score,metrics.roc_auc
0,xgboost,0.894147,0.850000,0.548387,0.666667,0.922202
1,random_forest,0.844334,0.579787,0.703226,0.635569,0.900796
2,logistic_regression,0.733499,0.394265,0.709677,0.506912,0.808831
3,logistic_regression,0.733499,0.394265,0.709677,0.506912,0.808831


**Observation (representative run):**

| Model | Accuracy | Precision | Recall | F1 | ROC-AUC |
|---|---|---|---|---|---|
| Logistic Regression | 0.733 | 0.394 | 0.710 | 0.507 | 0.809 |
| Random Forest | 0.848 | 0.598 | 0.652 | 0.623 | 0.896 |
| **XGBoost** | **0.897** | **0.867** | 0.548 | **0.672** | **0.925** |

**XGBoost** gives the best ROC-AUC and the best overall precision/F1 trade-off, so it is automatically selected as the best model, saved as `best_model.joblib`, and pushed to a Hugging Face **model repository**. MLflow tracked every hyperparameter and metric for all three runs, so the experiments are fully reproducible and comparable (run `mlflow ui` locally, inside the `tourism_project` folder, to view the tracking dashboard).

# Deployment

## Dockerfile

In [19]:
os.makedirs("tourism_project/deployment", exist_ok=True)

In [20]:
%%writefile tourism_project/deployment/Dockerfile
# Use a minimal base image with Python 3.9 installed
FROM python:3.9

# Set the working directory inside the container to /app
WORKDIR /app

# Copy all files from the current directory on the host to the container's /app directory
COPY . .

# Install Python dependencies listed in requirements.txt
RUN pip3 install --no-cache-dir -r requirements.txt

# Create a non-root user for security and switch to it
RUN useradd -m appuser
USER appuser

# Hugging Face Spaces expects the app to listen on port 7860
EXPOSE 7860

# Launch the Streamlit app
CMD ["streamlit", "run", "app.py", "--server.port=7860", "--server.address=0.0.0.0"]


Writing tourism_project/deployment/Dockerfile


## Streamlit App

Please ensure that the web app script is named `app.py`.

In [21]:
%%writefile tourism_project/deployment/app.py
"""Streamlit front-end for the Wellness Tourism Package prediction model."""
import os
import joblib
import pandas as pd
import streamlit as st
from huggingface_hub import hf_hub_download

from dotenv import load_dotenv
load_dotenv()

HF_USERNAME = os.getenv("HF_USERNAME", "<your-hf-username>")
MODEL_REPO_ID = f"{HF_USERNAME}/tourism-package-prediction-model"

@st.cache_resource
def load_model():
    model_path = hf_hub_download(repo_id=MODEL_REPO_ID, filename="best_model.joblib")
    return joblib.load(model_path)

st.set_page_config(page_title="Wellness Package Predictor", page_icon="\U0001F3DD\uFE0F")
st.title("\U0001F3DD\uFE0F Wellness Tourism Package \u2014 Purchase Predictor")
st.write("Enter the customer's details below to predict how likely they are to purchase the new Wellness Tourism Package.")

model = load_model()

col1, col2 = st.columns(2)
with col1:
    age = st.number_input("Age", min_value=18, max_value=100, value=35)
    type_of_contact = st.selectbox("Type of Contact", ["Self Enquiry", "Company Invited"])
    city_tier = st.selectbox("City Tier", [1, 2, 3])
    duration_of_pitch = st.number_input("Duration of Pitch (minutes)", min_value=0, max_value=60, value=15)
    occupation = st.selectbox("Occupation", ["Salaried", "Free Lancer", "Small Business", "Large Business"])
    gender = st.selectbox("Gender", ["Male", "Female"])
    num_persons = st.number_input("Number of Persons Visiting", min_value=1, max_value=10, value=2)
    num_followups = st.number_input("Number of Follow-ups", min_value=0, max_value=10, value=3)
    product_pitched = st.selectbox("Product Pitched", ["Basic", "Standard", "Deluxe", "Super Deluxe", "King"])
with col2:
    preferred_star = st.selectbox("Preferred Property Star", [3.0, 4.0, 5.0])
    marital_status = st.selectbox("Marital Status", ["Single", "Married", "Divorced"])
    num_trips = st.number_input("Number of Trips per Year", min_value=0, max_value=20, value=2)
    passport = st.selectbox("Holds Passport?", ["Yes", "No"])
    pitch_satisfaction = st.slider("Pitch Satisfaction Score", 1, 5, 3)
    own_car = st.selectbox("Owns a Car?", ["Yes", "No"])
    num_children = st.number_input("Number of Children Visiting (<5 yrs)", min_value=0, max_value=5, value=0)
    designation = st.selectbox("Designation", ["Executive", "Manager", "Senior Manager", "AVP", "VP"])
    monthly_income = st.number_input("Monthly Income", min_value=0, value=20000)

input_df = pd.DataFrame([{
    "Age": age, "TypeofContact": type_of_contact, "CityTier": city_tier,
    "DurationOfPitch": duration_of_pitch, "Occupation": occupation, "Gender": gender,
    "NumberOfPersonVisiting": num_persons, "NumberOfFollowups": num_followups,
    "ProductPitched": product_pitched, "PreferredPropertyStar": preferred_star,
    "MaritalStatus": marital_status, "NumberOfTrips": num_trips,
    "Passport": 1 if passport == "Yes" else 0, "PitchSatisfactionScore": pitch_satisfaction,
    "OwnCar": 1 if own_car == "Yes" else 0, "NumberOfChildrenVisiting": num_children,
    "Designation": designation, "MonthlyIncome": monthly_income,
}])

if st.button("Predict"):
    prediction = model.predict(input_df)[0]
    probability = model.predict_proba(input_df)[0][1]
    if prediction == 1:
        st.success(f"Likely to purchase the Wellness Package (probability: {probability:.1%})")
    else:
        st.warning(f"Unlikely to purchase the Wellness Package (probability: {probability:.1%})")


Writing tourism_project/deployment/app.py


## Dependency Handling

Please ensure that the dependency handling file is named `requirements.txt`.

In [22]:
%%writefile tourism_project/deployment/requirements.txt
streamlit==1.38.0
pandas==2.2.2
numpy==1.26.4
scikit-learn==1.5.1
xgboost==2.1.1
joblib==1.4.2
huggingface_hub==0.24.6


Writing tourism_project/deployment/requirements.txt


In [23]:
%%writefile tourism_project/deployment/README.md
---
title: Tourism Wellness Package Predictor
emoji: "\U0001F3DD\uFE0F"
colorFrom: blue
colorTo: green
sdk: docker
app_port: 7860
pinned: false
---

# Wellness Tourism Package Predictor

A Streamlit app that predicts whether a customer is likely to purchase the
"Visit with Us" Wellness Tourism Package, based on a trained scikit-learn /
XGBoost pipeline pulled from the Hugging Face Hub model repo.


Writing tourism_project/deployment/README.md


**Note:** A `README.md` with Hugging Face Space "front-matter" (the `sdk: docker` block above) is required for Spaces — it tells Hugging Face how to build and run the app.

# Hosting

In [26]:
%%writefile tourism_project/deployment/hosting.py
"""Creates/updates a Hugging Face Space and uploads the app files to it."""
import os
from huggingface_hub import HfApi, create_repo
from dotenv import load_dotenv
load_dotenv()

HF_TOKEN = os.getenv("HF_TOKEN")
HF_USERNAME = os.getenv("HF_USERNAME")
SPACE_REPO_ID = f"{HF_USERNAME}/tourism-package-prediction"

api = HfApi(token=HF_TOKEN)

create_repo(
    repo_id=SPACE_REPO_ID,
    repo_type="space",
    space_sdk="docker",
    private=False,
    exist_ok=True,
    token=HF_TOKEN,
)

api.upload_folder(
    folder_path="tourism_project/deployment",
    repo_id=SPACE_REPO_ID,
    repo_type="space",
    token=HF_TOKEN,
    allow_patterns=["app.py", "Dockerfile", "requirements.txt", "README.md"],
)

print(f"App deployed. Visit: https://huggingface.co/spaces/{SPACE_REPO_ID}")


Overwriting tourism_project/deployment/hosting.py


In [28]:
!python tourism_project/deployment/hosting.py

Traceback (most recent call last):
  File "c:\Users\gupta\OneDrive\Desktop\langchain-project\.venv\Lib\site-packages\huggingface_hub\utils\_http.py", line 836, in hf_raise_for_status
    response.raise_for_status()
  File "c:\Users\gupta\OneDrive\Desktop\langchain-project\.venv\Lib\site-packages\httpx2\_models.py", line 827, in raise_for_status
    raise HTTPStatusError(message, request=request, response=self)
httpx2.HTTPStatusError: Client error '402 Payment Required' for url 'https://huggingface.co/api/repos/create'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "c:\Users\gupta\OneDrive\Desktop\model dep\tourism_project\deployment\hosting.py", line 13, in <module>
    create_repo(
  File "c:\Users\gupta\OneDrive\Desktop\langchain-project\.venv\Lib\site-packages\huggingface_hub\utils\_validators.py", line 89, in _inner_fn
    return

**Observation:** This creates a public Hugging Face **Space** (Docker SDK) and uploads `app.py`, `Dockerfile`, `requirements.txt` and `README.md` to it. Hugging Face automatically builds the Docker image and launches the Streamlit app — no manual server setup required. Once the build finishes (a couple of minutes), the app is live at `https://huggingface.co/spaces/<your-username>/tourism-package-prediction`.

**\u2192 Paste your live Space link here once deployed:** `<add your Hugging Face Space link>`

# MLOps Pipeline with Github Actions Workflow

**Note:**

1. Before this workflow can run, add two **repository secrets** on GitHub (`Settings -> Secrets and variables -> Actions -> New repository secret`):
   - `HF_TOKEN` — your Hugging Face access token (WRITE permission)
   - `HF_USERNAME` — your Hugging Face username
2. The pipeline below has **4 jobs**, each depending on the previous one (`needs:`), so they run in sequence: `register-dataset -> data-prep -> model-training -> deploy-hosting`. This automates the exact same steps we ran manually above, so that every `git push` to `main` re-registers the data, re-cleans it, retrains and re-deploys the model automatically.

In [ ]:
%%writefile .github/workflows/pipeline.yml
name: Tourism Project Pipeline

on:
  push:
    branches:
      - main
  workflow_dispatch: {}

jobs:

  register-dataset:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - uses: actions/setup-python@v4
        with:
          python-version: "3.9"
      - name: Install Dependencies
        run: pip install -r tourism_project/requirements.txt
      - name: Upload Dataset to Hugging Face Hub
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
          HF_USERNAME: ${{ secrets.HF_USERNAME }}
        run: python tourism_project/model_building/data_register.py

  data-prep:
    needs: register-dataset
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - uses: actions/setup-python@v4
        with:
          python-version: "3.9"
      - name: Install Dependencies
        run: pip install -r tourism_project/requirements.txt
      - name: Clean, Split and Upload train/test data
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
          HF_USERNAME: ${{ secrets.HF_USERNAME }}
        run: python tourism_project/model_building/data_prep.py

  model-training:
    needs: data-prep
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - uses: actions/setup-python@v4
        with:
          python-version: "3.9"
      - name: Install Dependencies
        run: pip install -r tourism_project/requirements.txt
      - name: Train, Track with MLflow and Register Best Model
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
          HF_USERNAME: ${{ secrets.HF_USERNAME }}
        run: python tourism_project/model_building/train.py

  deploy-hosting:
    needs: model-training
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - uses: actions/setup-python@v4
        with:
          python-version: "3.9"
      - name: Install Dependencies
        run: pip install huggingface_hub==0.24.6
      - name: Deploy Streamlit App to Hugging Face Spaces
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
          HF_USERNAME: ${{ secrets.HF_USERNAME }}
        run: python tourism_project/deployment/hosting.py


**Note:** To use this YAML file for our use case, we need to:

1. Go to the GitHub repository for the project.
2. Create a folder named ***.github/workflows/*** (already created by the cell above).
3. In that folder, the file ***pipeline.yml*** already contains the content above.
4. Push the repository to GitHub (steps below) — GitHub Actions will then pick the workflow up automatically.

## Requirements file for the Github Actions Workflow

In [ ]:
%%writefile tourism_project/requirements.txt
pandas==2.2.2
numpy==1.26.4
scikit-learn==1.5.1
xgboost==2.1.1
mlflow==2.15.1
joblib==1.4.2
huggingface_hub==0.24.6


## Pushing the Project to GitHub

**Since we are working locally in VS Code (not Google Colab), run the commands below in the VS Code integrated terminal** (`` Ctrl+` `` / `` Cmd+` ``) from the root of your project folder, **not** as notebook cells — a local git repository is a one-time setup per machine, not something to redo on every notebook run.

1. Create a new, empty repository on GitHub (no README/gitignore) named e.g. `tourism-mlops-project`.
2. Generate a Personal Access Token: GitHub profile -> Settings -> Developer settings -> Personal access tokens -> Tokens (classic) -> Generate new token (scope: `repo`). Copy it somewhere safe — you'll paste it once as your password when pushing.
3. In the VS Code terminal:

```bash
cd tourism_project_root        # the folder containing tourism_project/ and .github/
git init
git config user.email "<your-github-email>"
git config user.name "<your-github-username>"
git remote add origin https://github.com/<your-github-username>/tourism-mlops-project.git
git add .
git commit -m "Initial commit: tourism MLOps pipeline"
git branch -M main
git push -u origin main
```

4. When prompted for a password, paste your **Personal Access Token** (not your GitHub password).
5. Go to the **Actions** tab of your GitHub repository — the `Tourism Project Pipeline` workflow should start running automatically on push.

**Observation:** Once pushed, GitHub Actions runs the 4 jobs in sequence in the cloud, fully reproducing the manual steps executed earlier in this notebook — the pipeline is now automated end-to-end: every future `git push` to `main` re-registers the data, re-preprocesses it, retrains the model with fresh MLflow tracking, and redeploys the Streamlit app to Hugging Face Spaces without any manual intervention.

# Output Evaluation

- GitHub (link to repository, screenshot of folder structure and executed workflow)

**GitHub repository link:** `<paste your GitHub repo link here>`

**To capture for submission:**
1. Screenshot of your repo's file/folder structure (showing `tourism_project/`, `.github/workflows/pipeline.yml`, etc.).
2. Screenshot of the **Actions** tab showing all 4 jobs (`register-dataset`, `data-prep`, `model-training`, `deploy-hosting`) completed with green checkmarks.

*(Insert screenshots as Markdown image cells here, e.g. `![repo structure](repo_structure.png)`, once captured.)*

- Streamlit on Hugging Face (link to HF space, screenshot of Streamlit app)

**Hugging Face Space link:** `<paste your Hugging Face Space link here>`

**To capture for submission:**
1. Screenshot of the live Streamlit app on Hugging Face Spaces, showing the input form.
2. Screenshot of a sample prediction (both a "Likely to purchase" and an "Unlikely to purchase" example are ideal).

*(Insert screenshots as Markdown image cells here once captured.)*

# Overall Insights and Conclusion

**Summary of the pipeline built in this notebook:**

- **Data Registration:** Raw `tourism.csv` (4,128 rows) registered to a Hugging Face dataset repo as the single source of truth.
- **Data Preparation:** Fixed `"Fe Male" -> "Female"` and `"Unmarried" -> "Single"` inconsistencies, dropped the non-predictive `CustomerID`/index columns, and created a stratified 80/20 train/test split (3,208 / 821 rows), preserving the ~19% positive-class rate.
- **Model Building & Tracking:** Compared Logistic Regression, Random Forest, and XGBoost inside a single `ColumnTransformer` + `Pipeline`, with every run's parameters and metrics logged to MLflow. **XGBoost was automatically selected as the best model** (ROC-AUC \u2248 0.925, Accuracy \u2248 0.897), and registered to a Hugging Face model repo.
- **Deployment:** Wrapped the model in a Streamlit app (`app.py`), containerized it with a `Dockerfile`, and hosted it on Hugging Face Spaces (Docker SDK), so business users can get live purchase-likelihood predictions without touching any code.
- **MLOps Pipeline:** A 4-stage GitHub Actions workflow (`register-dataset -> data-prep -> model-training -> deploy-hosting`) automates the entire pipeline on every `git push`, ensuring the model and app stay in sync with the latest data with no manual steps.

**Business takeaway:** Instead of manually and inconsistently guessing which customers to pitch the Wellness Package to, "Visit with Us" sales agents can now enter a customer's profile into the deployed app and get an instant, model-backed purchase-likelihood score — and because the whole pipeline is automated, the model can be refreshed as new customer data comes in, keeping recommendations current as customer behavior evolves.

<font size=6 color="navyblue">Power Ahead!</font>
___